In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
data = pd.read_csv('Final_Dataset_AIBD.csv')

C:\Users\apkar\AppData\Local\Temp\ipykernel_28768\639219182.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('Final_Dataset_AIBD.csv')


In [3]:
data

,Unnamed: 0.1,Unnamed: 0,Age,Gender,ICU,RUNDC3A,DEPDC1,BICDL1,ATP5PB,COTL1,...,CCR1,DMTN,CAVIN2,IZUMO4,SEPTIN10,ZCWPW1,DENND4C,TBX3,SCD5,H2AP
0,0,C1,39,male,no,0.410000,0.290000,3.500000,83.130000,793.950000,...,464.730000,24.290000,55.980000,1.790000,1.930000,2.260000,10.800000,0.000000,0.890000,0.0
1,1,C2,63,male,no,0.380000,0.930000,4.370000,94.440000,990.270000,...,177.660000,35.440000,99.140000,1.740000,1.690000,2.610000,6.190000,0.000000,1.130000,0.0
2,2,C3,33,male,no,0.390000,0.100000,2.240000,41.240000,719.210000,...,393.770000,6.810000,12.310000,1.860000,0.380000,6.120000,3.590000,0.000000,0.620000,0.0
3,3,C4,49,male,no,0.560000,0.190000,7.870000,123.220000,804.480000,...,256.230000,24.500000,122.510000,3.430000,1.820000,5.010000,9.480000,0.010000,1.590000,0.0
4,4,C5,49,male,no,0.450000,0.440000,1.920000,72.720000,505.980000,...,482.630000,15.100000,44.620000,0.510000,0.210000,3.150000,4.390000,0.000000,0.650000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180,180,NC81,18,male,NaN,2.535697,0.023387,1.061867,68.486990,878.511376,...,7.195875,18.339219,6.031346,4.281002,0.410692,2.368973,0.392739,0.000254,0.091454,0.0
181,181,NC82,15,male,NaN,0.890256,0.032683,7.413428,65.070772,1311.710166,...,71.612906,13.833811,29.663119,1.689109,0.852816,4.191334,2.895935,0.000254,0.917859,0.0
182,182,NC83,16,female,NaN,1.307012,0.023387,3.190243,50.016863,1346.659513,...,95.839170,19.880872,13.036078,1.895986,1.193758,4.573771,1.575842,0.000254,0.158958,0.0
183,183,NC84,14,male,NaN,1.138351,0.106631,9.613936,52.493368,1060.417768,...,41.763646,14.206949,25.061025,1.858951,0.404773,5.072356,2.224795,0.000254,0.500736,0.0


In [4]:
category = [i[0] for i in data['Unnamed: 0']]
print(category.count('N'))
print(category.count('C'))


85
100


In [5]:
from scipy import stats
from statsmodels.stats.multitest import multipletests

ALPHA = 0.05  # 95% significance level (FDR)

# Gene columns start after the metadata columns (..., Age, Gender, ICU)
gene_cols = list(data.columns[data.columns.get_loc('ICU') + 1:])
expr = data[gene_cols].apply(pd.to_numeric, errors='coerce')

# Group labels: 'C...' = case (C), 'N...' = control (N)
grp = data['Unnamed: 0'].astype(str).str[0]
case, ctrl = expr[grp == 'C'], expr[grp == 'N']

# Welch's t-test for every gene (two-sided)
t_stat, p_val = stats.ttest_ind(case, ctrl, equal_var=False, nan_policy='omit')

res = pd.DataFrame({
    'gene': gene_cols,
    'mean_case': case.mean().values,
    'mean_ctrl': ctrl.mean().values,
    't_stat': t_stat,
    'p_value': p_val,
})
res['pct_change'] = (res['mean_case'] - res['mean_ctrl']) / res['mean_ctrl'].replace(0, np.nan) * 100

# Benjamini-Hochberg FDR correction (genes with NaN p-values, e.g. constant genes, are excluded)
valid = res['p_value'].notna()
res['p_adj'] = np.nan
res.loc[valid, 'p_adj'] = multipletests(res.loc[valid, 'p_value'], alpha=ALPHA, method='fdr_bh')[1]
res['significant'] = res['p_adj'] < ALPHA
res['direction'] = np.where(~res['significant'], 'ns',
                            np.where(res['mean_case'] > res['mean_ctrl'], 'elevated', 'suppressed'))

n_tested = valid.sum()
n_sig = res['significant'].sum()
n_up = (res['direction'] == 'elevated').sum()
n_down = (res['direction'] == 'suppressed').sum()

print(f"Genes tested: {n_tested}")
print(f"Significant genes (BH-FDR < {ALPHA}): {n_sig} ({n_sig / n_tested * 100:.2f}%)")
print(f"  Elevated   : {n_up} ({n_up / n_tested * 100:.2f}%)")
print(f"  Suppressed : {n_down} ({n_down / n_tested * 100:.2f}%)")

sig = res[res['significant']]
print("\nMedian % change in mean expression among significant genes:")
print(sig.groupby('direction')['pct_change'].median())
sig.sort_values('p_adj').head(10)

Genes tested: 17721
Significant genes (BH-FDR < 0.05): 11619 (65.57%)
  Elevated   : 9894 (55.83%)
  Suppressed : 1725 (9.73%)

Median % change in mean expression among significant genes:
direction
elevated      56.000000
suppressed   -28.871949
Name: pct_change, dtype: float64


,gene,mean_case,mean_ctrl,t_stat,p_value,pct_change,p_adj,significant,direction
9973,UBE2C,10.8895,3.556923,14.206996,1.011315e-27,206.149438,1.792151e-23,True,elevated
4885,SLC15A3,29.3440,48.110769,-12.998533,4.811157e-27,-39.007419,4.262926e-23,True,suppressed
5163,SPC25,3.0064,0.903846,13.935956,3.774734e-26,232.622979,2.229735e-22,True,elevated
2165,PCLAF,6.5681,1.573846,13.733646,2.421592e-25,317.327957,1.072826e-21,True,elevated
326,KPNA2,24.9811,12.988462,12.700086,6.002588e-25,92.333017,2.127437e-21,True,elevated
5020,CDC25C,0.8179,0.195385,13.376469,8.587889e-25,318.610236,2.536433e-21,True,elevated
7688,CDCA8,2.9330,0.905000,13.147141,1.131103e-24,224.088398,2.863469e-21,True,elevated
14102,HNRNPLL,15.7489,7.198462,12.239777,3.966194e-24,118.781470,8.785616e-21,True,elevated
5014,NUSAP1,7.3458,1.595769,12.907097,1.901869e-23,360.329718,3.744780e-20,True,elevated
15196,GMNN,9.4490,3.920769,12.581148,2.299931e-23,140.998627,4.075708e-20,True,elevated


In [6]:
# ---- Median comparison: Mann-Whitney U test + Benjamini-Hochberg (FDR 5%) ----
# Re-uses `case`, `ctrl`, `gene_cols`, `ALPHA` from the previous cell
u_stat, p_mw = stats.mannwhitneyu(case, ctrl, alternative='two-sided', nan_policy='omit')

res_med = pd.DataFrame({
    'gene': gene_cols,
    'median_case': case.median().values,
    'median_ctrl': ctrl.median().values,
    'U_stat': u_stat,
    'p_value': p_mw,
})
res_med['pct_change'] = ((res_med['median_case'] - res_med['median_ctrl'])
                         / res_med['median_ctrl'].replace(0, np.nan) * 100)

valid_m = res_med['p_value'].notna()
res_med['p_adj'] = np.nan
res_med.loc[valid_m, 'p_adj'] = multipletests(res_med.loc[valid_m, 'p_value'],
                                              alpha=ALPHA, method='fdr_bh')[1]
res_med['significant'] = res_med['p_adj'] < ALPHA
res_med['direction'] = np.where(~res_med['significant'], 'ns',
                                np.where(res_med['median_case'] > res_med['median_ctrl'],
                                         'elevated', 'suppressed'))

n_tested_m = valid_m.sum()
n_sig_m = res_med['significant'].sum()
n_up_m = (res_med['direction'] == 'elevated').sum()
n_down_m = (res_med['direction'] == 'suppressed').sum()

print(f"Genes tested: {n_tested_m}")
print(f"Significant genes (Mann-Whitney U, BH-FDR < {ALPHA}): {n_sig_m} ({n_sig_m / n_tested_m * 100:.2f}%)")
print(f"  Elevated   : {n_up_m} ({n_up_m / n_tested_m * 100:.2f}%)")
print(f"  Suppressed : {n_down_m} ({n_down_m / n_tested_m * 100:.2f}%)")

sig_m = res_med[res_med['significant']]
print("\nMedian % change in median expression among significant genes:")
print(sig_m.groupby('direction')['pct_change'].median())
sig_m.sort_values('p_adj').head(10)

Genes tested: 17721
Significant genes (Mann-Whitney U, BH-FDR < 0.05): 12876 (72.66%)
  Elevated   : 9237 (52.12%)
  Suppressed : 3639 (20.53%)

Median % change in median expression among significant genes:
direction
elevated       55.240147
suppressed   -100.000000
Name: pct_change, dtype: float64


,gene,median_case,median_ctrl,U_stat,p_value,pct_change,p_adj,significant,direction
8605,CDC6,1.325,0.212232,8307.5,5.262420e-29,524.316902,9.325535e-25,True,elevated
14227,CCNB1,3.850,0.837814,8168.0,3.721703e-27,359.528991,9.843728e-24,True,elevated
2723,DEPDC1B,1.220,0.135421,8160.5,4.443870e-27,800.897463,9.843728e-24,True,elevated
17043,MCM10,0.490,0.046727,8187.0,2.076013e-27,948.634820,9.843728e-24,True,elevated
12025,MELK,1.235,0.146941,8183.0,2.260696e-27,740.474459,9.843728e-24,True,elevated
17562,CCNA2,3.335,0.684474,8201.0,1.376025e-27,387.235122,9.843728e-24,True,elevated
2341,ESCO2,0.630,0.090000,8161.0,4.328707e-27,600.000000,9.843728e-24,True,elevated
14916,PBK,1.025,0.059076,8154.5,3.737859e-27,1635.045869,9.843728e-24,True,elevated
5237,DTL,1.035,0.108234,8152.5,5.853308e-27,856.259347,1.152516e-23,True,elevated
5073,CHEK1,1.880,0.454672,8132.0,1.079411e-26,313.485088,1.594021e-23,True,elevated


In [7]:
import json

def export_genes(df, path):
    out = {
        'elevation': df.loc[df['direction'] == 'elevated', 'gene'].tolist(),
        'supression': df.loc[df['direction'] == 'suppressed', 'gene'].tolist(),
    }
    with open(path, 'w') as f:
        json.dump(out, f, indent=2)
    print(f"{path}: {len(out['elevation'])} elevation, {len(out['supression'])} supression")

export_genes(res, 'mean_significant_genes.json')
export_genes(res_med, 'median_significant_genes.json')

mean_significant_genes.json: 9894 elevation, 1725 supression
median_significant_genes.json: 9237 elevation, 3639 supression
